# Script 3 — Treinamento dos Modelos de ML (V3)
**TCC: Predição de Indicadores Financeiros Corporativos com ML e IA Generativa**

| Decisão | Justificativa |
|---------|---------------|
| **9 targets** | 3 primários (DRE) + 5 balanço + 1 caixa → habilita Z''-Score prospectivo completo |
| **Split temporal** | Treino ≤2022, Teste 2023–2024 — o modelo nunca viu dados futuros |
| **GroupKFold por empresa** | Evita vazamento temporal cruzado entre empresas no CV |
| **Log-transform seletivo** | Targets com skew>1 e mínimo>0 recebem log1p |
| **SMAPE como métrica principal** | Definido para negativos (Lucro Líquido pode ser negativo) |
| **Theil's U** | Prova que ML supera baseline ingênua |
| **Acurácia Direcional** | Percentual de acertos na direção (sobe/desce) |
| **Imputer no Pipeline** | Evita leakage de NaN das features YoY no CV |
| **Curvas de aprendizado** | Diagnóstico automático de overfitting/underfitting |
| **Análise de resíduos** | Valida pressupostos e detecta padrões sistemáticos |

## Etapa 0 — Dependências e configuração

In [ ]:
import logging, json, pickle, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import joblib
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import Ridge
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GroupKFold, GridSearchCV, learning_curve
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.impute import SimpleImputer

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:,.4f}'.format)

PASTA_SAIDA = Path('outputs')
PASTA_SAIDA.mkdir(exist_ok=True)

logger = logging.getLogger('pipeline_modelagem_v3')
logger.setLevel(logging.DEBUG)
logger.handlers.clear()
_fmt = logging.Formatter('%(asctime)s | %(levelname)-8s | %(message)s',
                         datefmt='%Y-%m-%d %H:%M:%S')
_sh = logging.StreamHandler(); _sh.setLevel(logging.INFO); _sh.setFormatter(_fmt)
logger.addHandler(_sh)
_fh = logging.FileHandler(PASTA_SAIDA / 'pipeline_modelagem_v3.log',
                          mode='w', encoding='utf-8')
_fh.setLevel(logging.DEBUG); _fh.setFormatter(_fmt)
logger.addHandler(_fh)

# ── Parâmetros ─────────────────────────────────────────────────────────────
N_SPLITS_EXT = 5
N_SPLITS_INT = 5
RANDOM_STATE = 42
ANO_CORTE    = 2022   # treino ≤ 2022, teste ≥ 2023

# Targets que recebem log1p (skew alto e mínimo > 0)
LOG_TARGETS = {
    'TARGET_DRE_3.01', 'TARGET_EBITDA',
    'TARGET_BPA_1', 'TARGET_BPA_1.01',
    'TARGET_BPP_2.01', 'TARGET_BPP_2.03', 'TARGET_BPP_2',
    'TARGET_DFC_MI_6.01',
}

logger.info("Script 3 V3 iniciado | sklearn=%s", __import__('sklearn').__version__)
print("✅ Dependências carregadas")


## Etapa 1 — Carregamento e split temporal

In [ ]:
dataset = pd.read_parquet(PASTA_SAIDA / 'dataset_preparado.parquet')

with open(PASTA_SAIDA / 'features.pkl',      'rb') as f: FEATURES      = pickle.load(f)
with open(PASTA_SAIDA / 'targets.pkl',       'rb') as f: TARGETS       = pickle.load(f)
with open(PASTA_SAIDA / 'kpis.pkl',          'rb') as f: KPIS          = pickle.load(f)
with open(PASTA_SAIDA / 'grupos_treino.pkl', 'rb') as f: GRUPOS_TREINO = pickle.load(f)

# ── Carregar splits do Script 2 (já com 3 conjuntos: treino/teste/prospectivo) ──
# O Script 2 grava treino.parquet e teste.parquet com o split temporal correto.
# O Script 3 consome esses arquivos diretamente — sem recriar o split —
# para garantir que treino ≤ 2022, teste 2023–2024, prospectivo ≥ 2025.
# Recriar o split aqui criaria inconsistência se o Script 2 tiver sido
# atualizado com novos dados (ex: ITR Q1/2026).
cam_treino = PASTA_SAIDA / 'treino.parquet'
cam_teste  = PASTA_SAIDA / 'teste.parquet'

if cam_treino.exists() and cam_teste.exists():
    treino = pd.read_parquet(cam_treino)
    teste  = pd.read_parquet(cam_teste)
    logger.info("Splits carregados dos parquets do Script 2")
else:
    # Fallback: Script 2 não foi rodado — recalcular com 3 conjuntos
    logger.warning("treino.parquet não encontrado — recalculando split temporal")
    if 'split' not in dataset.columns:
        dataset['split'] = np.where(
            dataset['ANO'].astype(float) <= ANO_CORTE, 'treino',
            np.where(
                dataset['ANO'].astype(float) <= 2024, 'teste',
                'prospectivo'
            )
        )
    treino = dataset[dataset['split'] == 'treino'].reset_index(drop=True)
    teste  = dataset[dataset['split'] == 'teste'].reset_index(drop=True)
    GRUPOS_TREINO = treino['CNPJ_CIA'].values
    treino.to_parquet(cam_treino, index=False)
    teste.to_parquet(cam_teste,   index=False)
    with open(PASTA_SAIDA / 'grupos_treino.pkl', 'wb') as f:
        pickle.dump(GRUPOS_TREINO, f)

# Validação
assert len(treino) > 0, "Treino vazio — verifique o Script 2"
assert len(teste)  > 0, "Teste vazio — verifique o Script 2"

# Verificar que o conjunto prospectivo NÃO vazou para treino ou teste
anos_treino = set(treino['ANO'].dropna().astype(int).unique())
anos_teste  = set(teste['ANO'].dropna().astype(int).unique())
anos_prosp  = {a for a in anos_treino | anos_teste if a >= 2025}
if anos_prosp:
    logger.error("Anos prospectivos vazaram para treino/teste: %s", anos_prosp)
else:
    logger.info("Isolamento prospectivo: PASSOU ✅ (nenhum ano ≥2025 no treino/teste)")

n_tot = len(treino) + len(teste)
logger.info("Treino: %d obs (≤%d) | Teste: %d obs (%d–%d)",
            len(treino), ANO_CORTE,
            len(teste),  ANO_CORTE+1, 2024)

print(f"\n{'='*65}")
print(f"  Split temporal — carregado do Script 2")
print(f"{'='*65}")
print(f"  Treino (≤{ANO_CORTE}): {len(treino):>4} obs | "
      f"DFP={(treino['ORIGEM']=='DFP').sum():>3} | "
      f"ITR={(treino['ORIGEM']=='ITR').sum():>3} | "
      f"anos {sorted(anos_treino)[:3]}...{sorted(anos_treino)[-1:]}")
print(f"  Teste ({ANO_CORTE+1}–2024): {len(teste):>4} obs | "
      f"DFP={(teste['ORIGEM']=='DFP').sum():>3} | "
      f"ITR={(teste['ORIGEM']=='ITR').sum():>3} | "
      f"anos {sorted(anos_teste)}")
print(f"  Prospectivo (≥2025): carregado pelo Script 5")
print(f"  Isolamento prospectivo: ✅")
print(f"{'='*65}")
print(f"\nFeatures: {len(FEATURES)} | Targets: {len(TARGETS)}")
print(f"Targets:")
for t in TARGETS:
    if t in treino.columns:
        log_flag = "📐log" if t in LOG_TARGETS else "   "
        n_tr = treino[t].notna().sum()
        n_te = teste[t].notna().sum() if t in teste.columns else 0
        print(f"  {log_flag} {t:<30} treino={n_tr:,} | teste={n_te:,}")


## Etapa 2 — Métricas e Baseline Ingênua

Além das métricas estatísticas clássicas, o pipeline calcula:

**Theil's U:** U<1 prova que o modelo supera a baseline de persistência.
**Acurácia Direcional:** % de acertos na direção (sobe/desce) — mais relevante para gestores.
**SMAPE:** erro percentual simétrico, definido mesmo para valores negativos (Lucro Líquido).

In [ ]:
def smape(y_true, y_pred):
    """Symmetric MAPE — definido para valores negativos e próximos de zero."""
    num   = np.abs(y_true - y_pred)
    denom = (np.abs(y_true) + np.abs(y_pred)) / 2
    mask  = denom > 0
    return float(np.mean(num[mask] / denom[mask])) if mask.sum() > 0 else np.nan


def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))


def theil_u(y_true, y_pred):
    """
    Coeficiente de desigualdade de Theil (U2).
    U < 1 → modelo supera a baseline ingênua de persistência.
    U = 1 → equivalente à baseline.
    U > 1 → pior que não fazer nada.
    """
    n = len(y_true)
    if n < 2:
        return np.nan
    # Baseline: y_pred = y_true[t-1] (persistência)
    erro_modelo   = np.sqrt(np.mean((y_true[1:] - y_pred[1:])**2))
    erro_baseline = np.sqrt(np.mean((y_true[1:] - y_true[:-1])**2))
    return float(erro_modelo / erro_baseline) if erro_baseline > 0 else np.nan


def acuracia_direcional(y_true, y_pred):
    """
    Percentual de acertos na direção (sobe/desce) em relação ao período anterior.
    Calculado sobre sequências ordenadas temporalmente.
    """
    if len(y_true) < 2:
        return np.nan
    dir_real = np.sign(np.diff(y_true))
    dir_pred = np.sign(np.diff(y_pred))
    mask = dir_real != 0
    return float(np.mean(dir_real[mask] == dir_pred[mask])) if mask.sum() > 0 else np.nan


def calcular_baseline(treino_df, teste_df, target):
    """Baseline ingênua: valor atual do período como previsão do próximo."""
    mapa = {
        'TARGET_DRE_3.01'    : 'DRE_3.01',
        'TARGET_DRE_3.11'    : 'DRE_3.11',
        'TARGET_EBITDA'      : 'EBITDA',
        'TARGET_BPA_1'       : 'BPA_1',
        'TARGET_BPA_1.01'    : 'BPA_1.01',
        'TARGET_BPP_2.01'    : 'BPP_2.01',
        'TARGET_BPP_2.03'    : 'BPP_2.03',
        'TARGET_BPP_2'       : 'BPP_2',
        'TARGET_DFC_MI_6.01' : 'DFC_MI_6.01',
    }
    col = mapa.get(target)
    if not col or col not in teste_df.columns:
        return {}
    mask = teste_df[target].notna() & teste_df[col].notna()
    if mask.sum() == 0:
        return {}
    y_t = teste_df.loc[mask, target].values
    y_p = teste_df.loc[mask, col].values
    return {
        'RMSE_baseline'  : rmse(y_t, y_p),
        'MAE_baseline'   : float(mean_absolute_error(y_t, y_p)),
        'SMAPE_baseline' : smape(y_t, y_p),
        'R2_baseline'    : float(r2_score(y_t, y_p)),
        'TheilU_baseline': 1.0,  # por definição, Theil da baseline = 1
        'DA_baseline'    : float(acuracia_direcional(y_t, y_p)),
    }


# Calcular baselines para todos os targets
baselines = {}
print("=== Baseline Ingênua (persistência) ===")
print(f"  {'Target':<30} {'RMSE':>14} {'SMAPE':>7} {'R²':>6} {'TheilU':>7} {'DA':>6}")
print(f"  {'-'*30} {'-'*14} {'-'*7} {'-'*6} {'-'*7} {'-'*6}")
for t in TARGETS:
    b = calcular_baseline(treino, teste, t)
    baselines[t] = b
    if b:
        print(f"  {t:<30} {b['RMSE_baseline']:>14,.0f} "
              f"{b['SMAPE_baseline']:>7.1%} {b['R2_baseline']:>6.3f} "
              f"{b['TheilU_baseline']:>7.2f} {b['DA_baseline']:>6.1%}")


## Etapa 3 — Algoritmos e grades de hiperparâmetros

In [ ]:
# GroupKFold por empresa
gkf_ext = GroupKFold(n_splits=N_SPLITS_EXT)
gkf_int = GroupKFold(n_splits=N_SPLITS_INT)

# ── Ridge ──────────────────────────────────────────────────────────────────
est_ridge = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler",  StandardScaler()),
    ("ridge",   Ridge()),
])
grade_ridge = {"ridge__alpha": [0.01, 0.1, 1.0, 10.0, 100.0, 1000.0]}

# ── SVR ────────────────────────────────────────────────────────────────────
est_svr = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler",  StandardScaler()),
    ("svr",     SVR(kernel="rbf", max_iter=20000)),
])
grade_svr = {
    "svr__C":       [0.1, 1.0, 10.0, 100.0],
    "svr__epsilon": [0.01, 0.05, 0.1, 0.5],
    "svr__gamma":   ["scale", "auto"],
}

# ── Random Forest ──────────────────────────────────────────────────────────
est_rf = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("rf",      RandomForestRegressor(n_estimators=300,
                                       random_state=RANDOM_STATE, n_jobs=-1)),
])
grade_rf = {
    "rf__max_depth":        [None, 5, 10, 20],
    "rf__min_samples_leaf": [1, 2, 5],
    "rf__max_features":     ["sqrt", "log2"],
}

# ── Gradient Boosting ──────────────────────────────────────────────────────
est_gb = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("gb",      GradientBoostingRegressor(random_state=RANDOM_STATE)),
])
grade_gb = {
    "gb__n_estimators":  [100, 200, 300],
    "gb__learning_rate": [0.01, 0.05, 0.1, 0.2],
    "gb__max_depth":     [3, 5],
    "gb__subsample":     [0.7, 0.8, 1.0],
}

ALGORITMOS = {
    "Ridge":            (est_ridge, grade_ridge),
    "SVR":              (est_svr,   grade_svr),
    "RandomForest":     (est_rf,    grade_rf),
    "GradientBoosting": (est_gb,    grade_gb),
}
logger.info("%d algoritmos | GroupKFold ext=%d int=%d",
            len(ALGORITMOS), N_SPLITS_EXT, N_SPLITS_INT)
print(f"✅ {len(ALGORITMOS)} algoritmos com GroupKFold(n={N_SPLITS_EXT})")


## Etapa 4 — Treinamento com Nested Cross-Validation

In [ ]:
def treinar_alg(nome, estimador, grade, X, y, grupos, gkf_int, gkf_ext,
                usar_log=False):
    """
    Nested CV com GroupKFold.

    Loop interno : GridSearchCV seleciona hiperparâmetros sem vazar empresas.
    Loop externo : estima generalização no espaço original (log revertido).

    Métricas calculadas: RMSE, MAE, SMAPE, R², Theil's U, Acurácia Direcional.
    """
    y_fit = np.log1p(y) if usar_log else y.copy()

    # ── Loop interno ────────────────────────────────────────────────────
    gs = GridSearchCV(
        estimador, grade,
        cv=list(gkf_int.split(X, y_fit, grupos)),
        scoring='neg_mean_squared_error',
        refit=True, n_jobs=-1, verbose=0,
    )
    gs.fit(X, y_fit)
    melhor = gs.best_estimator_

    # ── Loop externo ────────────────────────────────────────────────────
    rmse_v, mae_v, smape_v, r2_v, theil_v, da_v = [], [], [], [], [], []

    for tr_idx, val_idx in gkf_ext.split(X, y_fit, grupos):
        X_tr, X_val  = X[tr_idx], X[val_idx]
        y_tr         = y_fit[tr_idx]
        y_orig_val   = y[val_idx]

        melhor.fit(X_tr, y_tr)
        y_pred_raw = melhor.predict(X_val)
        y_pred     = np.expm1(y_pred_raw) if usar_log else y_pred_raw

        rmse_v.append(rmse(y_orig_val, y_pred))
        mae_v.append(float(mean_absolute_error(y_orig_val, y_pred)))
        smape_v.append(smape(y_orig_val, y_pred))
        r2_v.append(float(r2_score(y_orig_val, y_pred)))
        theil_v.append(theil_u(y_orig_val, y_pred))
        da_v.append(acuracia_direcional(y_orig_val, y_pred))

    # Fit final no conjunto completo de treino
    melhor.fit(X, y_fit)

    def _m(lst): return float(np.nanmean(lst))
    def _s(lst): return float(np.nanstd(lst))

    metricas = {
        'RMSE_CV'     : _m(rmse_v),   'RMSE_CV_std'  : _s(rmse_v),
        'MAE_CV'      : _m(mae_v),
        'SMAPE_CV'    : _m(smape_v),  'SMAPE_CV_std' : _s(smape_v),
        'R2_CV'       : _m(r2_v),     'R2_CV_std'    : _s(r2_v),
        'TheilU_CV'   : _m(theil_v),
        'DA_CV'       : _m(da_v),
        'log_transform': usar_log,
        'best_params' : gs.best_params_,
    }

    flag_theil = "✅" if metricas['TheilU_CV'] < 1 else "⚠️"
    logger.info("  %-20s RMSE=%10.0f±%8.0f  SMAPE=%5.1f%%  "
                "R²=%5.3f  TheilU=%s%.3f  DA=%.1f%%  log=%s",
                nome, metricas['RMSE_CV'], metricas['RMSE_CV_std'],
                metricas['SMAPE_CV']*100, metricas['R2_CV'],
                flag_theil, metricas['TheilU_CV'],
                metricas['DA_CV']*100, usar_log)
    print(f"  {flag_theil} {nome:<20} "
          f"RMSE={metricas['RMSE_CV']:>12,.0f}  "
          f"SMAPE={metricas['SMAPE_CV']:>5.1%}  "
          f"R²={metricas['R2_CV']:>6.3f}  "
          f"U={metricas['TheilU_CV']:.3f}  "
          f"DA={metricas['DA_CV']:.1%}")

    return melhor, metricas


# ── Execução ──────────────────────────────────────────────────────────────
resultados = {}

for target in TARGETS:
    usar_log = target in LOG_TARGETS
    b = baselines.get(target, {})

    print(f"\n{'='*72}")
    print(f"  TARGET: {target}  |  log={usar_log}")
    if b:
        print(f"  Baseline → RMSE={b.get('RMSE_baseline',0):,.0f}  "
              f"SMAPE={b.get('SMAPE_baseline',0):.1%}  "
              f"R²={b.get('R2_baseline',0):.3f}  "
              f"DA={b.get('DA_baseline',0):.1%}")
    print(f"  {'Alg':<22} {'RMSE':>14} {'SMAPE':>7} {'R²':>7} {'TheilU':>7} {'DA':>6}")
    print(f"  {'-'*22} {'-'*14} {'-'*7} {'-'*7} {'-'*7} {'-'*6}")

    # Filtrar obs com target não-nulo
    df_t = treino[FEATURES + [target]].copy()
    df_t = df_t[df_t[target].notna()].reset_index(drop=True)
    mask = treino[target].notna()
    grupos_t = GRUPOS_TREINO[mask.values]

    X = df_t[FEATURES].values
    y = df_t[target].values

    resultados[target] = {}
    for nome, (est, grade) in ALGORITMOS.items():
        modelo, metricas = treinar_alg(
            nome, est, grade, X, y, grupos_t,
            gkf_int, gkf_ext, usar_log=usar_log,
        )
        resultados[target][nome] = (modelo, metricas)
        joblib.dump({'modelo': modelo, 'log_transform': usar_log,
                     'features': FEATURES},
                    PASTA_SAIDA / f'modelo_{target}_{nome}.pkl')

    logger.info("TARGET %s concluído", target)


## Etapa 5 — Avaliação no conjunto de teste hold-out

In [ ]:
def avaliar_teste(modelo, X_te, y_te, usar_log):
    y_pred_raw = modelo.predict(X_te)
    y_pred = np.expm1(y_pred_raw) if usar_log else y_pred_raw
    mask = np.isfinite(y_te) & np.isfinite(y_pred)
    yt, yp = y_te[mask], y_pred[mask]
    return {
        'RMSE_teste'  : rmse(yt, yp),
        'MAE_teste'   : float(mean_absolute_error(yt, yp)),
        'SMAPE_teste' : smape(yt, yp),
        'R2_teste'    : float(r2_score(yt, yp)),
        'TheilU_teste': theil_u(yt, yp),
        'DA_teste'    : acuracia_direcional(yt, yp),
    }


print("\n=== Avaliação no Teste Hold-out (2023–2024) ===")
metricas_teste = {}

for target in TARGETS:
    usar_log = target in LOG_TARGETS
    b = baselines.get(target, {})
    df_te = teste[FEATURES + [target]].copy()
    df_te = df_te[df_te[target].notna()]
    X_te  = df_te[FEATURES].values
    y_te  = df_te[target].values

    metricas_teste[target] = {}
    baseline_rmse = b.get('RMSE_baseline', np.inf)

    print(f"\n{target}  (baseline RMSE={baseline_rmse:,.0f}  "
          f"DA={b.get('DA_baseline',0):.1%})")
    print(f"  {'Algoritmo':<20} {'RMSE':>14} {'SMAPE':>7} "
          f"{'R²':>7} {'TheilU':>7} {'DA':>6} {'Bateu?':>7}")
    print(f"  {'-'*20} {'-'*14} {'-'*7} {'-'*7} {'-'*7} {'-'*6} {'-'*7}")

    for nome, (modelo, _) in resultados[target].items():
        m = avaliar_teste(modelo, X_te, y_te, usar_log)
        metricas_teste[target][nome] = m
        bateu  = m['RMSE_teste'] < baseline_rmse
        theil_ok = m['TheilU_teste'] < 1.0 if m['TheilU_teste'] else False
        flag = "✅" if bateu and theil_ok else ("🟡" if bateu else "❌")
        print(f"  {flag} {nome:<18} {m['RMSE_teste']:>14,.0f} "
              f"{m['SMAPE_teste']:>7.1%} {m['R2_teste']:>7.3f} "
              f"{m['TheilU_teste']:>7.3f} {m['DA_teste']:>6.1%} "
              f"{'✅' if bateu else '❌':>7}")
        logger.info("Teste | %s | %s: RMSE=%.0f SMAPE=%.2f%% R2=%.3f TheilU=%.3f DA=%.1f%%",
                    target, nome, m['RMSE_teste'], m['SMAPE_teste']*100,
                    m['R2_teste'], m['TheilU_teste'], m['DA_teste']*100)


## Etapa 6 — Feature Importance

In [ ]:
def extrair_importancia(modelo, features, nome_alg):
    step = [s for s,_ in modelo.steps][-1]
    est_final = modelo.named_steps[step]
    if hasattr(est_final, 'feature_importances_'):
        imp = est_final.feature_importances_
    elif hasattr(est_final, 'coef_'):
        imp = np.abs(est_final.coef_)
    else:
        return pd.Series(dtype=float)
    return pd.Series(imp, index=features).sort_values(ascending=False)


feature_importances = {}
print("\n=== Feature Importance — Melhor Modelo por Target ===")

n_t = len(TARGETS)
fig, axes = plt.subplots(n_t, 1, figsize=(11, 5*n_t))
if n_t == 1: axes = [axes]

for i, target in enumerate(TARGETS):
    melhor_nome = max(metricas_teste[target],
                      key=lambda n: metricas_teste[target][n]['R2_teste'])
    melhor_mod  = resultados[target][melhor_nome][0]
    imp = extrair_importancia(melhor_mod, FEATURES, melhor_nome)
    feature_importances[target] = {'algoritmo': melhor_nome,
                                    'importancias': imp.to_dict()}

    if not imp.empty:
        top = imp.head(min(12, len(imp)))
        colors = ['#1f4e79' if v == top.values[0] else
                  '#2e75b6' if v >= top.values[0]*0.7 else '#9dc3e6'
                  for v in top.values[::-1]]
        axes[i].barh(range(len(top)), top.values[::-1], color=colors, alpha=0.9)
        axes[i].set_yticks(range(len(top)))
        axes[i].set_yticklabels(top.index[::-1], fontsize=9)
        axes[i].set_title(f"{target.replace('TARGET_','')} — {melhor_nome} "
                           f"(R²={metricas_teste[target][melhor_nome]['R2_teste']:.3f})",
                           fontsize=11, fontweight='bold')
        axes[i].set_xlabel('Importância Relativa')
        axes[i].grid(axis='x', alpha=0.3)
        for j, v in enumerate(top.values[::-1]):
            axes[i].text(v + imp.max()*0.005, j, f'{v:.3f}', va='center', fontsize=8)
    print(f"  {target}: {melhor_nome} | top3={list(imp.head(3).index)}")

plt.suptitle('Feature Importance — Melhor Modelo por Target',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(PASTA_SAIDA / 'feature_importance.png', dpi=150, bbox_inches='tight')
plt.close()
print("  ✅ Salvo: feature_importance.png")


## Etapa 7 — Curvas de Aprendizado

In [ ]:
print("\nGerando curvas de aprendizado...")

n_t = len(TARGETS)
fig, axes = plt.subplots(1, n_t, figsize=(7*n_t, 5))
if n_t == 1: axes = [axes]

for i, target in enumerate(TARGETS):
    usar_log = target in LOG_TARGETS
    df_t = treino[FEATURES + [target]].copy()
    df_t = df_t[df_t[target].notna()].reset_index(drop=True)
    mask = treino[target].notna()
    grupos_t = GRUPOS_TREINO[mask.values]
    X = df_t[FEATURES].values
    y = np.log1p(df_t[target].values) if usar_log else df_t[target].values

    melhor_nome = max(metricas_teste[target],
                      key=lambda n: metricas_teste[target][n]['R2_teste'])
    melhor_mod  = resultados[target][melhor_nome][0]

    try:
        sizes, tr_sc, val_sc = learning_curve(
            melhor_mod, X, y,
            cv=list(gkf_ext.split(X, y, grupos_t)),
            scoring='r2',
            train_sizes=np.linspace(0.2, 1.0, 6),
            n_jobs=-1,
        )
        ax = axes[i]
        ax.plot(sizes, tr_sc.mean(1), 'o-', label='Treino',  color='#1f4e79', lw=2)
        ax.fill_between(sizes, tr_sc.mean(1)-tr_sc.std(1),
                         tr_sc.mean(1)+tr_sc.std(1), alpha=0.12, color='#1f4e79')
        ax.plot(sizes, val_sc.mean(1), 's--', label='Validação', color='#c0392b', lw=2)
        ax.fill_between(sizes, val_sc.mean(1)-val_sc.std(1),
                         val_sc.mean(1)+val_sc.std(1), alpha=0.12, color='#c0392b')
        gap  = tr_sc.mean(1)[-1] - val_sc.mean(1)[-1]
        diag = ('overfitting' if gap > 0.15 else
                'underfitting' if val_sc.mean(1)[-1] < 0.3 else 'OK')
        ax.set_title(f"{target.replace('TARGET_','')}\n{melhor_nome}",
                      fontsize=10, fontweight='bold')
        ax.set_xlabel(f'Tamanho do treino  |  Gap={gap:.2f} → {diag}', fontsize=9)
        ax.set_ylabel('R²')
        ax.legend(fontsize=9)
        ax.grid(alpha=0.3)
        logger.info("Curva %s/%s: gap=%.3f diag=%s", target, melhor_nome, gap, diag)
    except Exception as e:
        logger.warning("Curva de aprendizado falhou %s/%s: %s", target, melhor_nome, e)
        axes[i].text(0.5,0.5,'Erro na curva\n'+str(e)[:60],
                     ha='center',va='center',transform=axes[i].transAxes,fontsize=9)

plt.suptitle('Curvas de Aprendizado — Diagnóstico de Bias/Variância',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(PASTA_SAIDA / 'curvas_aprendizado.png', dpi=150, bbox_inches='tight')
plt.close()
print("  ✅ Salvo: curvas_aprendizado.png")


## Etapa 8 — Análise de Resíduos

In [ ]:
print("\nGerando análise de resíduos...")

n_t = len(TARGETS)
fig, axes = plt.subplots(n_t, 2, figsize=(14, 5*n_t))
if n_t == 1: axes = axes.reshape(1,-1)

for i, target in enumerate(TARGETS):
    usar_log = target in LOG_TARGETS
    df_te = teste[FEATURES + [target]].copy()
    df_te = df_te[df_te[target].notna()]
    X_te  = df_te[FEATURES].values
    y_te  = df_te[target].values

    melhor_nome = max(metricas_teste[target],
                      key=lambda n: metricas_teste[target][n]['R2_teste'])
    mod = resultados[target][melhor_nome][0]
    y_pred = np.expm1(mod.predict(X_te)) if usar_log else mod.predict(X_te)
    residuos = y_te - y_pred

    # Predito × Observado
    ax1 = axes[i,0]
    lim = max(np.nanmax(np.abs(y_te)), np.nanmax(np.abs(y_pred))) * 1.05
    ax1.scatter(y_pred, y_te, alpha=0.45, s=18, color='#1f4e79', edgecolors='none')
    ax1.plot([0, lim], [0, lim], 'r--', lw=1.5)
    ax1.set_xlabel('Predito (R$ mil)')
    ax1.set_ylabel('Observado (R$ mil)')
    ax1.set_title(f"{target.replace('TARGET_','')} — {melhor_nome}\nPredito × Observado",
                   fontsize=10, fontweight='bold')
    r2_val = metricas_teste[target][melhor_nome]['R2_teste']
    ax1.text(0.05, 0.92, f'R²={r2_val:.3f}', transform=ax1.transAxes, fontsize=9,
             bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

    # Resíduos × Predito
    ax2 = axes[i,1]
    ax2.scatter(y_pred, residuos, alpha=0.45, s=18, color='#744210', edgecolors='none')
    ax2.axhline(0, color='r', lw=1.5, ls='--')
    ax2.axhline(np.std(residuos), color='gray', lw=1, ls=':', alpha=0.7)
    ax2.axhline(-np.std(residuos), color='gray', lw=1, ls=':', alpha=0.7)
    ax2.set_xlabel('Predito (R$ mil)')
    ax2.set_ylabel('Resíduo (R$ mil)')
    ax2.set_title(f'Resíduos  |  skew={pd.Series(residuos).skew():.2f}  '
                   f'σ={np.std(residuos):,.0f}', fontsize=10, fontweight='bold')

plt.suptitle('Análise de Resíduos — Conjunto de Teste 2023–2024',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(PASTA_SAIDA / 'analise_residuos.png', dpi=150, bbox_inches='tight')
plt.close()
print("  ✅ Salvo: analise_residuos.png")


## Etapa 9 — Persistência completa

In [ ]:
rows_cv, rows_te = [], []

for target, algs in resultados.items():
    b = baselines.get(target, {})
    for alg, (_, m) in algs.items():
        rows_cv.append({'Target':target,'Algoritmo':alg,
                        'RMSE_CV':m['RMSE_CV'],'RMSE_CV_std':m.get('RMSE_CV_std'),
                        'SMAPE_CV':m['SMAPE_CV'],'R2_CV':m['R2_CV'],
                        'TheilU_CV':m.get('TheilU_CV'),'DA_CV':m.get('DA_CV'),
                        'log_transform':m['log_transform'],
                        'best_params':str(m['best_params'])})
        mt = metricas_teste[target][alg]
        rows_te.append({'Target':target,'Algoritmo':alg,
                        'RMSE_teste':mt['RMSE_teste'],'MAE_teste':mt['MAE_teste'],
                        'SMAPE_teste':mt['SMAPE_teste'],'R2_teste':mt['R2_teste'],
                        'TheilU_teste':mt.get('TheilU_teste'),
                        'DA_teste':mt.get('DA_teste'),
                        'RMSE_baseline':b.get('RMSE_baseline'),
                        'Bateu_baseline': mt['RMSE_teste'] < b.get('RMSE_baseline', np.inf),
                        'TheilU_ok': (mt.get('TheilU_teste',1.0) or 1.0) < 1.0})

df_cv   = pd.DataFrame(rows_cv)
df_te   = pd.DataFrame(rows_te)
melhores = {t: df_te[df_te['Target']==t].sort_values('R2_teste',ascending=False)
              .iloc[0]['Algoritmo'] for t in TARGETS}

df_cv.to_csv(PASTA_SAIDA / 'resultados_cv.csv',    index=False)
df_te.to_csv(PASTA_SAIDA / 'resultados_teste.csv', index=False)

with open(PASTA_SAIDA / 'resultados_cv.pkl',       'wb') as f: pickle.dump(resultados, f)
with open(PASTA_SAIDA / 'metricas_teste.pkl',      'wb') as f: pickle.dump(metricas_teste, f)
with open(PASTA_SAIDA / 'baselines.pkl',           'wb') as f: pickle.dump(baselines, f)
with open(PASTA_SAIDA / 'feature_importances.pkl', 'wb') as f: pickle.dump(feature_importances, f)
with open(PASTA_SAIDA / 'melhores_modelos.pkl',    'wb') as f: pickle.dump(melhores, f)

relatorio = {
    'versao'       : 'V3_9targets_split_temporal',
    'ano_corte'    : ANO_CORTE,
    'n_treino'     : int(len(treino)),
    'n_teste'      : int(len(teste)),
    'algoritmos'   : list(ALGORITMOS.keys()),
    'targets'      : TARGETS,
    'log_targets'  : list(LOG_TARGETS),
    'features'     : FEATURES,
    'melhores'     : melhores,
    'baselines'    : {t:{k:float(v) for k,v in b.items()
                         if isinstance(v,(int,float,np.floating))}
                      for t,b in baselines.items()},
}
with open(PASTA_SAIDA / 'relatorio_modelagem.json','w',encoding='utf-8') as f:
    json.dump(relatorio, f, indent=2, ensure_ascii=False, default=str)

print("\n" + "═"*72)
print("  RESUMO FINAL — Script 3 V3")
print("═"*72)
print(f"  Treino  : {len(treino):,} obs (≤{ANO_CORTE}) | "
      f"DFP={(treino['ORIGEM']=='DFP').sum()} | ITR={(treino['ORIGEM']=='ITR').sum()}")
print(f"  Teste   : {len(teste):,} obs (≥{ANO_CORTE+1}) | "
      f"DFP={(teste['ORIGEM']=='DFP').sum()}  | ITR={(teste['ORIGEM']=='ITR').sum()}")
print(f"  Modelos : {len(TARGETS) * len(ALGORITMOS)} ({len(TARGETS)} targets × {len(ALGORITMOS)} algoritmos)")
print(f"  Targets : {len(TARGETS)} (3 primários + 5 balanço + 1 caixa)")
print(f"  Métricas: RMSE, MAE, SMAPE, R², Theil's U, Acurácia Direcional")
print(f"  Melhores por R²:")
for t, alg in melhores.items():
    m = metricas_teste[t][alg]
    print(f"    {t:<35} {alg:<20} R²={m['R2_teste']:.3f}  "
          f"SMAPE={m['SMAPE_teste']:.1%}  U={m['TheilU_teste']:.3f}")
print("═"*72)
print("  ✅ Pronto para o Script 2.1 (EDA) e Script 4 (Avaliação + Z'')")
print("═"*72)
